# Training Pipeline - Part 3: BERT Transformer Training
## Cyber AI Agent v2.0.0

**Purpose:** Train transformer-based classifier using BERT (Layer 4)

**Technique:** Transfer learning with fine-tuning

**Inputs:**
- X_train_scaled.pkl (raw features for NLP text generation)
- y_train.pkl
- X_test_scaled.pkl
- y_test.pkl

**Outputs:**
- bert_classifier/ (fine-tuned model directory)
- bert_metrics.json

---

## 1. Setup and Data Loading

In [1]:
import torch
import numpy as np
import pandas as pd
import joblib
import json
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# Load preprocessed data
X_train = joblib.load('../datasets/X_train_scaled.pkl')
X_test = joblib.load('../datasets/X_test_scaled.pkl')
y_train = joblib.load('../datasets/y_train.pkl')
y_test = joblib.load('../datasets/y_test.pkl')

print(f"✅ Data loaded: {X_train.shape[0]} training, {X_test.shape[0]} test samples")

c:\Users\yasiru\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'datasets'

## 2. Generate NLP Text from Network Features

In [ ]:
print("\n📝 GENERATING NLP DESCRIPTIONS")
print("=" * 80)

def generate_nlp_description(row):
    """Convert numeric network features to semantic text."""
    text = f"""Network flow analysis: Duration {int(max(row.get('Flow Duration', 0), 1))}ms. 
    Forward packets: {int(max(row.get('Total Fwd Packets', 0), 0))} packets, 
    {int(max(row.get('Total Length of Fwd Packets', 0), 0))} bytes. 
    Backward packets: {int(max(row.get('Total Backward Packets', 0), 0))} packets, 
    {int(max(row.get('Total Length of Bwd Packets', 0), 0))} bytes. 
    Flow rate: {row.get('Flow Bytes/s', 0):.2f} bytes/s, 
    Packet rate: {row.get('Flow Packets/s', 0):.2f} packets/s. 
    Target port: {int(row.get('Destination Port', 0))}. 
    Average packet size: {row.get('Average Packet Size', 0):.2f} bytes."""
    return ' '.join(text.split())  # Remove extra whitespace

# Generate text descriptions
train_texts = []
for idx, row in X_train.iterrows():
    text = generate_nlp_description(row)
    train_texts.append(text)

test_texts = []
for idx, row in X_test.iterrows():
    text = generate_nlp_description(row)
    test_texts.append(text)

print(f"\n1. Generated {len(train_texts)} training descriptions")
print(f"2. Generated {len(test_texts)} test descriptions")
print(f"\n3. Example training text:")
print(f"   {train_texts[0][:150]}...")

## 3. Prepare BERT Dataset

In [ ]:
print("\n🔄 PREPARING BERT DATASET")
print("=" * 80)

# Load pre-trained BERT tokenizer
model_name = 'prajjwal1/bert-tiny'  # Lightweight BERT variant (4M params)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"\n1. Loaded tokenizer: {model_name}")
print(f"   Vocab size: {tokenizer.vocab_size}")

# Create datasets
train_dataset = Dataset.from_dict({
    'text': train_texts,
    'labels': y_train.values
})

test_dataset = Dataset.from_dict({
    'text': test_texts,
    'labels': y_test.values
})

print(f"\n2. Created training dataset: {len(train_dataset)} samples")
print(f"   Created test dataset: {len(test_dataset)} samples")

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

print(f"\n3. Tokenizing datasets...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
test_tokenized = test_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

print(f"   ✅ Tokenization complete")
print(f"   Max sequence length: 128 tokens")

## 4. Load Pre-trained BERT and Fine-tune

In [ ]:
print("\n🤖 FINE-TUNING BERT")
print("=" * 80)

# Load pre-trained BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5  # 5 attack types
)

model = model.to(device)

print(f"\n1. Loaded pre-trained BERT:")
print(f"   Model: {model_name}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Device: {device}")

# Define training arguments
training_args = TrainingArguments(
    output_dir='../trained_models/bert_classifier',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy='epoch',
    logging_steps=100,
    seed=42
)

print(f"\n2. Training configuration:")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Batch size: 32")
print(f"   Epochs: 3")
print(f"   Max sequence length: 128")

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

print(f"\n3. Starting fine-tuning...")
import time
start_time = time.time()

# Train
trainer.train()

training_time = time.time() - start_time
print(f"\n✅ Fine-tuning complete in {training_time:.2f} seconds")

## 5. Evaluate BERT Model

In [ ]:
print("\n📊 BERT EVALUATION")
print("=" * 80)

# Get predictions
predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\n1. Overall Performance:")
print(f"   Accuracy: {accuracy:.4f}")
print(f"   Precision (weighted): {precision:.4f}")
print(f"   Recall (weighted): {recall:.4f}")
print(f"   F1-Score (weighted): {f1:.4f}")

print(f"\n2. Classification Report:")
print(classification_report(y_test, y_pred))

## 6. Save Model and Metrics

In [ ]:
import os

print("\n💾 SAVING BERT MODEL")
print("=" * 80)

os.makedirs('../trained_models/bert_classifier', exist_ok=True)

# Save fine-tuned model
model.save_pretrained('../trained_models/bert_classifier')
tokenizer.save_pretrained('../trained_models/bert_classifier')

print(f"\n1. Saved fine-tuned BERT:")
print(f"   Location: ../trained_models/bert_classifier/")
print(f"   Includes: model weights, config, tokenizer")

# Save metrics
metrics = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'training_time': float(training_time),
    'test_samples': int(len(y_test)),
    'model_type': 'BERT Transformer (Transfer Learning)',
    'base_model': model_name,
    'num_classes': 5,
    'max_sequence_length': 128,
    'epochs': 3
}

with open('../trained_models/bert_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n2. Saved metrics:")
print(f"   bert_metrics.json")
print(f"   {metrics}")

print(f"\n✅ BERT training complete!")
print(f"\nNext: Run 04_autoencoder.ipynb for unsupervised learning")